In [163]:
import pandas as pd
import numpy as np
from math import ceil
import json

Loads the random sample

In [164]:
data = pd.read_csv('data_rs.csv')

Keep the needed fields

In [165]:
chosen = data[['avisoid', 'avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo']]
chosen['avisoid'] = chosen['avisoid'].astype(int)

/tmp/ipykernel_6532/926371173.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chosen['avisoid'] = chosen['avisoid'].astype(int)


Turns everyting to string

In [166]:
chosen['avisocargo'] = chosen['avisocargo'].astype(str)
chosen['avisocuerpo'] = chosen['avisocuerpo'].astype(str)
chosen['disponibilidadnombre'] = chosen['disponibilidadnombre'].astype(str)
chosen['avisorequisitos'] = chosen['avisorequisitos'].astype(str)
chosen['avisolugartrabajo'] = chosen['avisolugartrabajo'].astype(str)

/tmp/ipykernel_6532/1442197346.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chosen['avisocargo'] = chosen['avisocargo'].astype(str)
/tmp/ipykernel_6532/1442197346.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chosen['avisocuerpo'] = chosen['avisocuerpo'].astype(str)
/tmp/ipykernel_6532/1442197346.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://

Assign a group to each ad

In [167]:
groups = 50
ads = len(chosen)
print("Ads per person: ", ads/groups)

Ads per person:  9.96


In [168]:
# Step 1: Duplicate the DataFrame to have the same advertisements at least twice (in at least two groups)
chosen = pd.concat([chosen, chosen], ignore_index=True)
len(chosen)

996

In [169]:
# Step 2: Create a list of group numbers ensuring each group has at least 50 advertisements
group_numbers = list(range(1, groups + 1))
len(group_numbers)


50

In [170]:
# Step 3: Shuffle the list to distribute the groups randomly
np.random.shuffle(group_numbers)

In [171]:
# Step 4: Assign the shuffled group numbers to the DataFrame using the modulo operator
chosen['groups'] = [group_numbers[i % len(group_numbers)] for i in range(len(chosen))]

Here we start preparing the JSON resource for the app, the key is the 'avisoid' variable and the value of the item is a list  in this order of the other variables ['avisocargo', 'avisocuerpo', 'disponibilidadnombre', 'avisorequisitos', 'avisolugartrabajo']

In [172]:
chosen['list'] = chosen.apply(lambda row: [row[col] for col in ['avisocargo', 'avisocuerpo', 'disponibilidadnombre',
       'avisorequisitos', 'avisolugartrabajo']], axis=1)

In [180]:
chosen = chosen.sort_values('groups')

In [189]:
# Group by 'groups' and construct the nested dictionary
grouped = chosen.groupby('groups').apply(lambda x: dict(zip(x['avisoid'], x['list']))).to_dict()
len(grouped)

/tmp/ipykernel_6532/1966960847.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = chosen.groupby('groups').apply(lambda x: dict(zip(x['avisoid'], x['list']))).to_dict()


50

In [196]:
with open('jobads4.json', 'w') as json_file:
    json.dump(grouped, json_file, indent=4)